<a href="https://colab.research.google.com/github/fischaaulia/DataWineQuality/blob/main/004_UTS_PDAB_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Preprocessing**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

**Load Dataset**

In [ ]:
train_df = pd.read_csv('data_training.csv')
test_df = pd.read_csv('data_testing.csv')

In [ ]:
train_df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
count,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000,857.000000
mean,8.261960,0.529393,0.267351,2.506184,0.086830,15.782964,45.978413,0.996692,3.313092,0.656709,10.430338,5.653442,813.749125
std,1.701992,0.179162,0.195144,1.293512,0.048721,10.300402,31.692113,0.001901,0.152079,0.167364,1.066971,0.821777,463.807063
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.390000,8.400000,3.000000,0.000000
25%,7.100000,0.395000,0.090000,1.900000,0.070000,7.000000,21.000000,0.995520,3.210000,0.550000,9.500000,5.000000,413.000000
50%,7.900000,0.520000,0.250000,2.200000,0.079000,14.000000,38.000000,0.996680,3.310000,0.620000,10.200000,6.000000,814.000000
75%,9.100000,0.640000,0.420000,2.600000,0.090000,21.000000,63.000000,0.997800,3.400000,0.730000,11.100000,6.000000,1214.000000
max,15.600000,1.580000,1.000000,15.500000,0.611000,68.000000,278.000000,1.003200,4.010000,2.000000,14.000000,8.000000,1597.000000


In [ ]:
test_df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,Id
count,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000
mean,8.458392,0.537168,0.271399,2.609965,0.087241,15.113636,45.723776,0.996846,3.304790,0.660699,10.477389,778.660839
std,1.873036,0.181229,0.201552,1.527564,0.042696,10.100644,35.909202,0.001994,0.169794,0.179456,1.127771,464.383455
min,5.000000,0.180000,0.000000,1.200000,0.034000,3.000000,7.000000,0.990840,2.860000,0.330000,8.400000,2.000000
25%,7.100000,0.392500,0.090000,1.900000,0.071000,7.000000,21.000000,0.995605,3.200000,0.560000,9.600000,402.250000
50%,8.000000,0.530000,0.250000,2.200000,0.081000,12.000000,35.000000,0.996760,3.295000,0.620000,10.200000,747.000000
75%,9.400000,0.650000,0.427500,2.700000,0.091000,21.000000,55.750000,0.998100,3.400000,0.720000,11.175000,1169.500000
max,15.900000,1.330000,0.760000,15.400000,0.415000,68.000000,289.000000,1.003690,4.010000,1.950000,14.900000,1590.000000


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 857 entries, 0 to 856
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         857 non-null    float64
 1   volatile acidity      857 non-null    float64
 2   citric acid           857 non-null    float64
 3   residual sugar        857 non-null    float64
 4   chlorides             857 non-null    float64
 5   free sulfur dioxide   857 non-null    float64
 6   total sulfur dioxide  857 non-null    float64
 7   density               857 non-null    float64
 8   pH                    857 non-null    float64
 9   sulphates             857 non-null    float64
 10  alcohol               857 non-null    float64
 11  quality               857 non-null    int64  
 12  Id                    857 non-null    int64  
dtypes: float64(11), int64(2)
memory usage: 87.2 KB


Dataset ini terdiri dari 857 baris data training dengan rata-rata kualitas anggur berada di angka 5.65 (skala 3-8). Terlihat bahwa variabel memiliki rentang nilai yang sangat beragam, misalnya, total sulfur dioxide memiliki nilai maksimal hingga 278, sementara density hanya di kisaran 1.0. Perbedaan skala yang drastis ini membuktikan bahwa memerlukan Feature Scaling agar model tidak bias terhadap variabel dengan angka besar.

# **Data Cleaning**

**Cek Missing Values**

In [ ]:
# Data training
missing_values = train_df.isnull().sum()
print(missing_values)

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
Id                      0
dtype: int64


In [ ]:
# Data testing
missing_values = test_df.isnull().sum()
print(missing_values)

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
Id                      0
dtype: int64


In [ ]:
# Penanganan jika ada nilai yang hilang
# Menggunakan median karena lebih tahan terhadap outlier
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

Berdasarkan pengecekan menggunakan isnull().sum(), tidak ditemukan nilai kosong pada dataset training maupun testing. Namun, sebagai langkah preventif, saya tetap menerapkan fungsi pengisian nilai menggunakan Median. Hal ini dilakukan untuk menjamin model tetap berjalan stabil jika di masa mendatang terdapat data baru yang memiliki nilai hilang.

In [ ]:
# Pengecekan statistik memastikan tidak ada '0' lagi
print("\nStatistik Deskriptif : \n", train_df.describe())
print("\nStatistik Deskriptif : \n", test_df.describe())


Statistik Deskriptif : 
        fixed acidity  volatile acidity  citric acid  residual sugar  \
count     857.000000        857.000000   857.000000      857.000000   
mean        8.261960          0.529393     0.267351        2.506184   
std         1.701992          0.179162     0.195144        1.293512   
min         4.600000          0.120000     0.000000        0.900000   
25%         7.100000          0.395000     0.090000        1.900000   
50%         7.900000          0.520000     0.250000        2.200000   
75%         9.100000          0.640000     0.420000        2.600000   
max        15.600000          1.580000     1.000000       15.500000   

        chlorides  free sulfur dioxide  total sulfur dioxide     density  \
count  857.000000           857.000000            857.000000  857.000000   
mean     0.086830            15.782964             45.978413    0.996692   
std      0.048721            10.300402             31.692113    0.001901   
min      0.012000             

**Memisahkan Fitur dan Target**

In [ ]:
x = train_df.drop(columns=['Id','quality'])
y = train_df['quality']
x_test_final = test_df.drop(columns=['Id'])

# **Feature Scaling**

In [ ]:
# Inisialisasi scaler
scaler = StandardScaler()

x_scaled = scaler.fit_transform(x)
x_test_final_scaled = scaler.transform(x_test_final)

Menggunakan StandardScaler untuk menstandarisasi fitur sehingga memiliki rata-rata 0 dan standar deviasi 1. Langkah ini krusial untuk menyeimbangkan pengaruh setiap variabel kimiawi.

**Split data untuk evaluasi internal**

In [ ]:
# Membagi data: 80% untuk latihan, 20% untuk ujian mandiri (validasi)
x_train, x_val, y_train, y_val = train_test_split(x_scaled, y, test_size=0.2, random_state=42)

Pemisahan ini bertujuan untuk menguji performa model pada data yang belum pernah diliat sebelumnya sebelum melakukan prediksi final.

# **Modeling**

In [ ]:
# Membuat model Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)

**Melatih model pada data training**

In [ ]:
model.fit(x_train, y_train)

RandomForestClassifier(random_state=42)

**Melakukan prediksi pada data validasi**

In [ ]:
y_pred_val = model.predict(x_val)
print(f"Akurasi Model pada Data Validasi: {accuracy_score(y_val, y_pred_val) * 100:.2f}%")

Akurasi Model pada Data Validasi: 58.14%


Hasil pengujian pada data validasi menunjukkan akurasi sebesar 58.14%. Meskipun angka ini terlihat sedang, hasil ini tergolong cukup baik mengingat dataset ini memiliki rentang kelas kualitas yang cukup rapat (skala 3-8). Dengan angka tersebut, model ini terbukti sudah mampu mengenali pola dasar dan keterkaitan antara fitur-fitur kimiawi, seperti kadar alkohol dan tingkat keasaman terhadap nilai kualitas anggur yang diberikan.

Untuk pengembangan selanjutnya, akurasi dapat ditingkatkan dengan melakukan Hyperparameter Tuning atau mencoba teknik penanganan data tidak seimbang (imbalance data) pada kelas kualitas tertentu.

# **Prediksi dan Tabel Hasil**

**Melakukan prediksi pada data testing yang sudah di-scale sebelumnya**

In [ ]:
final_predictions = model.predict(x_test_final_scaled)

Model yang telah dilatih kemudian digunakan untuk memprediksi 286 data pada data_testing.

**Membuat DataFrame**

In [ ]:
df_hasil = pd.DataFrame({
    'Id': test_df['Id'],
    'quality': final_predictions })

**Simpan ke file CSV**

In [ ]:
nama_file = 'hasilprediksi_004.csv'
df_hasil.to_csv(nama_file, index=False)

print(f"DataFrame telah disimpan ke dalam file CSV: {nama_file}")

DataFrame telah disimpan ke dalam file CSV: hasilprediksi_004.csv


Meskipun data asli memiliki rentang kualitas 3-8, hasil prediksi pada data testing cenderung terkonsentrasi pada rentang 5-7. Hal ini disebabkan oleh Data Imbalance, di mana sampel untuk kualitas 5 dan 6 sangat mendominasi dataset training. Akibatnya, model Random Forest lebih cenderung memprediksi kelas mayoritas tersebut untuk meminimalkan risiko kesalahan (error), sehingga kelas ekstrem seperti 3, 4, dan 8 lebih jarang muncul dalam hasil prediksi final.